**Testing the App and Documenting Results:** Test the app with a few real questions and document the results.

In [1]:
%pip install chromadb sentence-transformers google-genai python-dotenv

Defaulting to user installation because normal site-packages is not writeable
Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.3 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [2]:
import os
import time
import chromadb
from sentence_transformers import SentenceTransformer
from google import genai
from google.genai.errors import APIError
from dotenv import load_dotenv

load_dotenv()
api_key = os.getenv("GEMINI_API_KEY")

if not api_key:
    raise SystemExit("No GEMINI_API_KEY found.")

client = genai.Client(api_key=api_key)
embedding_model = SentenceTransformer("all-MiniLM-L6-v2")

db_client = chromadb.PersistentClient(path="./week8_rag_db")
collection = db_client.get_or_create_collection(name="week8_documents")

print("Documents currently stored:", collection.count())

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Documents currently stored: 6


In [3]:
# Same functions as Notebook 02 (copied here so this notebook is self-contained).

def retrieve_context(question, top_k=3, max_distance=1.1):
    if collection.count() == 0:
        return None

    question_embedding = embedding_model.encode(question).tolist()
    results = collection.query(query_embeddings=[question_embedding], n_results=top_k)

    chunks = results["documents"][0]
    distances = results["distances"][0]

    if not chunks or min(distances) > max_distance:
        return None

    return "\n".join(f"Context {i}: {chunk}" for i, chunk in enumerate(chunks, start=1))


def ask_gemini(question, context=None):
    if context:
        prompt = f"""
Answer the question using the retrieved context when it is relevant.
If the context does not contain the answer, use your general knowledge.

Retrieved context:
{context}

Question:
{question}

Give a concise and accurate answer.
"""
    else:
        prompt = f"""
No relevant information was found in the user's documents for this question.
Answer using your general knowledge, and briefly mention that this answer is
not based on the user's own documents.

Question:
{question}
"""

    model = "gemini-3.6-flash"

    for attempt in range(3):
        try:
            response = client.models.generate_content(model=model, contents=prompt)
            if response.text:
                return response.text.strip()
            return "Gemini returned an empty answer."
        except APIError as error:
            error_text = str(error)
            if "503" in error_text or "UNAVAILABLE" in error_text:
                wait_seconds = 2 ** attempt
                print(f"{model} is busy. Retrying in {wait_seconds} seconds...")
                time.sleep(wait_seconds)
            else:
                return f"Gemini API error: {error}"

    return "gemini-3.6-flash is temporarily unavailable. Please try again later."

## Test questions
Change these questions to match your own documents from Notebook 01.

In [4]:
test_questions = [
    "What is Streamlit used for?",              # covered by sample docs
    "What does a try/except block do?",           # covered by sample docs
    "What is the capital of Japan?",               # NOT covered, tests fallback
    "Explain what a README file is for.",          # covered by sample docs
]

In [5]:
results = []

for question in test_questions:
    print("=" * 70)
    print("QUESTION:", question)

    try:
        context = retrieve_context(question)
        context_found = context is not None
        answer = ask_gemini(question, context)
    except Exception as error:
        # Even in testing, one broken question should not stop the rest.
        context_found = False
        answer = f"ERROR while answering: {error}"

    print("Context found:", context_found)
    print("Answer:", answer)

    results.append({
        "question": question,
        "context_found": context_found,
        "answer": answer
    })

QUESTION: What is Streamlit used for?
Context found: True
Answer: Based on the provided context, Streamlit is used to turn a normal Python script into a simple web application.
QUESTION: What does a try/except block do?
Context found: True
Answer: Based on the provided context, a try/except block allows a program to catch an error and keep running instead of crashing.
QUESTION: What is the capital of Japan?
Context found: False
Answer: Please note that this answer is based on general knowledge, as no relevant information was found in your documents. 

The capital of Japan is **Tokyo**.
QUESTION: Explain what a README file is for.
Context found: True
Answer: Based on the provided context, a README file explains what a project does and provides instructions on how someone else can run it.


## Save the results to a text file
This file is the "documented results" the Week 8 deliverable asks for.

In [ ]:
with open("week8_test_results.txt", "w", encoding="utf-8") as file:
    file.write("Week 8 RAG App - Test Results\n")
    file.write("=" * 40 + "\n\n")

    for item in results:
        file.write(f"Question: {item['question']}\n")
        file.write(f"Context found: {item['context_found']}\n")
        file.write(f"Answer: {item['answer']}\n")
        file.write("-" * 40 + "\n")

print("Saved results to week8_test_results.txt")

Saved results to week8_test_results.txt
